# Analysis of Emergency Obstetric Care (EmOC) in Harare, Zimbabwe
> Note: This notebook requires the [environment dependencies](requirements.txt) to be installed
> as well as either an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local instance of the ORS server.

## Model Summary:

This notebook provides the means to generate a dataset that is described in the [model documentation](../kano/dataset-interpretability.md).

## Workflow Summary:

The notebook gives an overview of the distribution of centres offering EmOC in the city, their classification and how they can be accessed during an emergency. Open source data from OpenStreetMap and tools (such as the openrouteservice) were used to create accessibility measures. Spatial analysis and other data analytics functions led to generating outputs within the 100x100m grid cells that categorised them into three levels: low, medium, and high.

* **Preprocessing**: Get data for EmOC facilities.
* **Analysis for Offer**:
    * Filter or classify EmOC facilities based on discussed criteria.
    * Visualise EmOC faccilities in their categories.
* **Analysis for Accessibility**:
    * Compute travel times to facilities using openrouteservice API or other routing services.
    * Generate areas for low, medium and high categories based on discussed criteria.
* **Analysis for Demmand**:
    * Downscale the popluation data to the 100x100m grid cells.
    * Derive socio-economic descriptors based on discussed criteria.

* **Result**: Generate results as GIS-compatible files.


### Datasets and Tools:
* [openrouteservice](https://openrouteservice.org/) - generate isochrones on the OpenStreetMap road network

#  Workflow

Make sure you have the required packages installed. You can install them using pip:

```bash
pip install -r requirements.txt
```

This study integrates various Python geospatial analysis libraries and packages to support spatial data processing, visualization, and isochrone generation. The os module is used to interact with the operating system, managing file paths and reading environment variables such as API keys. folium library along with its MarkerCluster plugin, facilitates the creation of interactive maps for visualizing large-scale geospatial data. The openrouteservice.client serves as an interface to the OpenRouteService API, enabling the extraction of isochrones. pandas library for data analysis, provides functions for analyzing, cleaning, exploring, and manipulating data, while fiona supports reading and writing real-world data using multi-layered GIS formats, such as shapefiles. The shapely package is employed for the manipulation and analysis of planar geometric objects.

## Setting up the virtual environment

```bash
# Create a new virtual environment
python -m venv .venv
activate .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [1]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd


import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point

from pathlib import Path
from shapely.geometry import Polygon

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler

### Setting up the public API Key from OpenRouteService
In this study, users must obtain an ORS Matrix API key from the [OpenRouteService](https://openrouteservice.org/) platform and subsequently interacted with the OpenRouteService API through the instantiation of the OpenRouteService client. This is the OpenRouteService [API documentation](https://openrouteservice.org/dev/#/api-docs/introduction) for ORS Core-Version 9.0.0. 

Generate a [API Key](https://openrouteservice.org/dev/#/home?tab=1) (Token) it is necessary to sign up at the OpenRouteService dashboard by using your E-mail address or sign up with your GitHub. After logging in, go to the Dashboard by clicking on your profile icon and navigate to the API Keys section. Click "Create API Key" to generate a free key and then choose a service plan (the free plan has limited requests per day). Copy the API Key and store it securely. 

OpenRouteService primarily uses API keys for authentication. However, if a token is required for certain endpoints, you can send a request with your API key in the Authorization header. This process facilitated various geospatial analysis functions, including isochrone generation.


### Option 1: Using an ORS API Key
Make sure you have a .env file in the root directory with the following content:
```bash
    OPENROUTESERVICE_API_KEY='your_api_key'
```

In [ ]:
# Read the api key from the .env file
%load_ext dotenv
%dotenv
api_key = os.getenv('OPENROUTESERVICE_API_KEY')
client = openrouteservice.Client(key=api_key)

### Setting up relevant processing folders

There are different data sources used across the notebook. To handle these data sets, it is recommended to use three directories for input, temp and output data. Some of the files are related to healthcare facilities, population data. The healthcare facilities data is usualy the result of gathering global or national datasets and then carrying out local validation according to the local context. 

Despite being official, administrative boundaries may not reflect the actual patterns of human settlement or economic activity. Therefore, the team used the Functional Urban Area (FUA) as a complementary definition of the study areas. The FUA is defined by [the Joint Research Centre of the European Commission](https://commission.europa.eu/about/departments-and-executive-agencies/joint-research-centre_en) as the actual urban sprawl and human activities, encompassing the core city and economically or socially integrated surrounding regions. The FUA was obtained from [the Global Human Settlement Layer (GHSL) ](https://human-settlement.emergency.copernicus.eu/)dataset, which provides spatial data for functional urban areas worldwide. 

The following datasets are considered as input data for the analysis:


* [Datasets of health facilities](../scripts/Kano/data-inputs/healthcare_facilities.geojson)
* [Population: Women in childbearing age](../scripts/Kano/data-inputs/kano_nga_f_15_49_2015_1km.tif) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447)
* [Study Area](../../../docs/study-areas/grid-boundary-kano.gpkg) defined by the IDEAMAPS team

In [2]:
# Set paths to access Harare data
# Define directories
data_inputs = '../scripts/Harare/data-inputs/'
data_temp = '../scripts/Harare/data-temp/'
model_outputs = '../harare/'

## 1. Data Collection

### Validated healthcare facilities - (Supply/Offer)
For Kano, the classification for validation was determined with the assistance of local experts, based on data obtained from the [datasets of health facilities](https://doi.org/10.6084/m9.figshare.22689667.v2).

In [25]:
healthcare_facilities_validated = gpd.read_file(data_inputs + 'healthcare_facilities_harare.geojson')
healthcare_facilities_validated

,Longitude,Latitude,Facility_name,Province,District,Ward,Ownership_type,Ownership,EmOC_status,Services_offered,Facility_level,Facility_type,geometry
0,31.098854,-18.028461,St.Michael's (Chihumbiri),Harare,Chitungwiza,23,Private,Independent practice,Not designated,Basic outpatient & maternal/child health services,Primary,Private clinic,POINT (31.09885 -18.02846)
1,31.098963,-18.028547,"Michael's 24 Hour Accident, Emergency & Maternity",Harare,Chitungwiza,23,Private,Independent practice,Not designated,Full obstetric & surgical services,Secondary,Private hospital,POINT (31.09896 -18.02855)
2,31.070139,-18.014130,Citimed/Chitungwiza South Med Hospital,Harare,Chitungwiza,13,Private,Group / Corporate owned,Not designated,Full obstetric & surgical services,Secondary,Private hospital,POINT (31.07014 -18.01413)
3,31.070454,-17.998762,Chidodo Clinic,Harare,Chitungwiza,14,Public,Local council,Not designated,Basic outpatient & maternal/child health services,Primary,Municipal clinic,POINT (31.07045 -17.99876)
4,31.211887,-17.825977,Tafara Family 24 Clinic,Harare,Harare,46,Private,Independent practice,Not designated,Basic outpatient & maternal/child health services,Primary,Private clinic,POINT (31.21189 -17.82598)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,31.052044,-17.790347,Health First Family Clinic - Dr J Chagonda,Harare,Harare,7,Private,Independent practice,Not designated,Basic outpatient & maternal/child health services,Primary,Private clinic,POINT (31.05204 -17.79035)
128,31.048776,-17.791989,Dr. D Djordjevic,Harare,Harare,7,Private,Independent practice,Comprehensive,Full obstetric & surgical services,Secondary,Private medical center,POINT (31.04878 -17.79199)
129,31.056000,-17.782323,University of Zimbabwe Students Clinic,Harare,Harare,17,Public,Government / Ministry of Higher and Tertiary E...,Not designated,Basic outpatient & maternal/child health services,Primary,Service clinic,POINT (31.056 -17.78232)
130,31.034116,-17.800039,Health mode Pharmacy,Harare,Harare,7,Private,Group / Corporate owned,Not applicable,Pharmaceutical dispensing & counseling services,Primary,Pharmacy,POINT (31.03412 -17.80004)


In [26]:
healthcare_facilities_validated["EmOC_status"].unique()

array(['Not designated', 'Not applicable', 'Basic', 'Comprehensive'],
      dtype=object)

In [27]:
healthcare_facilities_validated = healthcare_facilities_validated.copy()

healthcare_facilities_validated = healthcare_facilities_validated[
    healthcare_facilities_validated["EmOC_status"].isin(["Basic", "Comprehensive"])
]

healthcare_facilities_validated["local_validation"] = (
    healthcare_facilities_validated["Ownership_type"].str.capitalize()
    + " "
    + healthcare_facilities_validated["EmOC_status"]
    + " EmOC"
)

healthcare_facilities_validated["local_validation"].value_counts()

local_validation
Public Basic EmOC             17
Public Comprehensive EmOC      3
Private Comprehensive EmOC     1
Name: count, dtype: int64

In [28]:
healthcare_facilities_validated['hcf_id'] = range(len(healthcare_facilities_validated))
healthcare_facilities_validated

,Longitude,Latitude,Facility_name,Province,District,Ward,Ownership_type,Ownership,EmOC_status,Services_offered,Facility_level,Facility_type,geometry,local_validation,hcf_id
13,30.994402,-17.884174,Highfields Polyclinic,Harare,Harare,25,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (30.9944 -17.88417),Public Basic EmOC,0
16,31.035328,-17.859606,Mbare Polyclinic,Harare,Harare,11,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (31.03533 -17.85961),Public Basic EmOC,1
22,30.950854,-17.907724,Glenview Polyclinic,Harare,Harare,31,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (30.95085 -17.90772),Public Basic EmOC,2
29,31.062272,-18.017600,Chitungwiza General Hospital,Harare,Chitungwiza,13,Public,Government / Ministry of Health and Child Care...,Comprehensive,Full obstetric & surgical services,Tertiary,General hospital,POINT (31.06227 -18.0176),Public Comprehensive EmOC,3
38,30.970587,-17.854463,Kambuzuma Polyclinic,Harare,Harare,14,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (30.97059 -17.85446),Public Basic EmOC,4
42,31.196645,-17.835547,Mabvuku Polyclinic,Harare,Harare,20,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (31.19664 -17.83555),Public Basic EmOC,5
45,31.086431,-17.878907,Hatfield Polyclinic,Harare,Harare,22,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (31.08643 -17.87891),Public Basic EmOC,6
50,31.107381,-17.697793,Hatcliffe Polyclinic,Harare,Harare,Peri-urban/Informal,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (31.10738 -17.69779),Public Basic EmOC,7
52,31.012382,-17.860447,Harare Central Hospital,Harare,Harare,13,Public,Government / Ministry of Health and Child Care...,Comprehensive,Full obstetric & surgical services,Quaternary,Central hospital,POINT (31.01238 -17.86045),Public Comprehensive EmOC,8
55,30.935376,-17.891499,Budiriro Polyclinic,Harare,Harare,33,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (30.93538 -17.8915),Public Basic EmOC,9


In [29]:
healthcare_facilities_validated.to_file(data_inputs + 'healthcare_facilities_harare_emoc.geojson', driver='GeoJSON')

### Population Grid Data (Demand)
This data originally comes as a grid (1km resolution) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447) to transform it into a 100x100m grid, we use a procedure explained below. 

Note: explain the process to scale down the population data. 
note: explain the rational for female population between 15-49 years old.

In [11]:
study_area = gpd.read_file(data_inputs + 'grid-boundary-harare.gpkg')
raster_path = data_inputs + 'zwe_f_15_49_2015_1km.tif'

Clipping the population data to our study area

In [12]:
with rasterio.open(raster_path) as dataset:
    geometries = [study_area.geometry.unary_union.__geo_interface__]
    clipped_image, clipped_transform = mask(dataset, geometries, crop=True)
    band1 = clipped_image[0] # Read the first band of the raster

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_85583/2284915905.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometries = [study_area.geometry.unary_union.__geo_interface__]


In [13]:
out_meta = dataset.meta.copy()
out_meta.update({
        "height": clipped_image.shape[1],
        "width": clipped_image.shape[2],
        "transform": clipped_transform
    })

In [14]:
with rasterio.open(data_inputs + 'harare_zwe_f_15_49_2015_1km.tif', "w", **out_meta) as dest:
    dest.write(clipped_image)

### Adding population data at 1km grid to 100m grid

In [15]:
# reading in geotiff file as numpy array
def read_tif(file: Path):
    if not file.exists():
        raise FileNotFoundError(f'File {file} not found')

    with rasterio.open(file) as dataset:
        arr = dataset.read()  # (bands X height X width)
        nodata = dataset.nodata
        transform = dataset.transform
        crs = dataset.crs

    # Replace NoData value with NaN
    if nodata is not None:
        arr[arr == nodata] = np.nan

    return arr.transpose((1, 2, 0)), transform, crs

def raster2vector(arr, transform, crs) -> gpd.GeoDataFrame:
    height, width, bands = arr.shape

    # Generate pixel coordinates
    geometries = []
    pixel_values = []

    for row in range(height):
        for col in range(width):
            x_min, y_max = transform * (col, row)  # Top-left corner
            x_max, y_min = transform * (col + 1, row + 1)  # Bottom-right corner

            pixel_value = arr[row, col].tolist()[0]  # Convert numpy array to list
            polygon = Polygon([(x_min, y_max), (x_max, y_max), (x_max, y_min), (x_min, y_min)])

            geometries.append(polygon)
            pixel_values.append(pixel_value)

    # Convert to DataFrame
    gdf = gpd.GeoDataFrame({'pop_grid_pop': pixel_values, 'geometry': geometries}, crs=crs)

    return gdf

epsg = 'EPSG:32736' # UTM for harare 

In [16]:
# Preparing grid
grid_file = data_inputs + 'grid-boundary-harare.gpkg'
grid = gpd.read_file(grid_file)
grid = grid.to_crs(epsg)
grid['grid_id'] = range(len(grid))
grid = grid[['grid_id', 'geometry','latitude', 'lat_min', 'lat_max', 'longitude', 'lon_min','lon_max']].set_geometry('geometry')
grid

,grid_id,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,"POLYGON ((302882.989 8053010.803, 302872.771 8...",-17.600408,-17.600820,-17.599996,31.141838,31.141280,31.142395
1,1,"POLYGON ((302992.081 8053011.873, 302981.862 8...",-17.600408,-17.600820,-17.599996,31.142865,31.142308,31.143423
2,2,"POLYGON ((303101.172 8053012.942, 303090.954 8...",-17.600408,-17.600820,-17.599996,31.143893,31.143335,31.144451
3,3,"POLYGON ((303210.263 8053014.01, 303200.046 80...",-17.600408,-17.600820,-17.599996,31.144921,31.144363,31.145479
4,4,"POLYGON ((303319.355 8053015.078, 303309.137 8...",-17.600408,-17.600820,-17.599996,31.145949,31.145391,31.146506
...,...,...,...,...,...,...,...,...
175895,175895,"POLYGON ((301479.313 8002914.13, 301468.826 80...",-18.052853,-18.053266,-18.052441,31.123882,31.123322,31.124441
175896,175896,"POLYGON ((301588.301 8002915.236, 301577.814 8...",-18.052853,-18.053266,-18.052441,31.124911,31.124351,31.125471
175897,175897,"POLYGON ((301697.289 8002916.341, 301686.802 8...",-18.052853,-18.053266,-18.052441,31.125940,31.125381,31.126500
175898,175898,"POLYGON ((301806.277 8002917.445, 301795.79 80...",-18.052853,-18.053266,-18.052441,31.126970,31.126410,31.127530


Building footprint data is used to estimate population distribution within each 1km cell. We recommend using open-source building footprint data from the [Overture Map Foundation](https://overturemaps.org/). Building centroids are spatially joined to a 100 m resolution grid, and the number of buildings within each 100 m cell (bcount) is subsequently calculated.

In [17]:
# Count buildings per grid cell

# Load Google building footprints
building_file = data_inputs + 'harare_GOB.parquet' # check building-footprints.ipynb to get building footprints
buildings = gpd.read_parquet(building_file)
buildings = buildings.to_crs(epsg)
buildings['centroid'] = buildings['geometry'].centroid

for df in [grid, buildings]:
    cols_to_remove = [c for c in df.columns if c.startswith('index_') or c.endswith('_left') or c.endswith('_right')]
    if cols_to_remove:
        df.drop(columns=cols_to_remove, inplace=True)

# Join buildings to grid using centroid
grid_buildings = grid.sjoin(
    buildings.set_geometry('centroid').drop(columns='geometry'),
    how='inner',
    predicate='intersects'
)

# Count buildings per grid cell
building_counts = grid_buildings.groupby('grid_id').size().rename('bcount')

# Add building count to grid
grid = grid.merge(building_counts, on='grid_id', how='left')
grid['bcount'] = grid['bcount'].fillna(0)   # assign 0 to empty cells
grid

,grid_id,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max,bcount
0,0,"POLYGON ((302882.989 8053010.803, 302872.771 8...",-17.600408,-17.600820,-17.599996,31.141838,31.141280,31.142395,16.0
1,1,"POLYGON ((302992.081 8053011.873, 302981.862 8...",-17.600408,-17.600820,-17.599996,31.142865,31.142308,31.143423,0.0
2,2,"POLYGON ((303101.172 8053012.942, 303090.954 8...",-17.600408,-17.600820,-17.599996,31.143893,31.143335,31.144451,0.0
3,3,"POLYGON ((303210.263 8053014.01, 303200.046 80...",-17.600408,-17.600820,-17.599996,31.144921,31.144363,31.145479,0.0
4,4,"POLYGON ((303319.355 8053015.078, 303309.137 8...",-17.600408,-17.600820,-17.599996,31.145949,31.145391,31.146506,1.0
...,...,...,...,...,...,...,...,...,...
175895,175895,"POLYGON ((301479.313 8002914.13, 301468.826 80...",-18.052853,-18.053266,-18.052441,31.123882,31.123322,31.124441,0.0
175896,175896,"POLYGON ((301588.301 8002915.236, 301577.814 8...",-18.052853,-18.053266,-18.052441,31.124911,31.124351,31.125471,0.0
175897,175897,"POLYGON ((301697.289 8002916.341, 301686.802 8...",-18.052853,-18.053266,-18.052441,31.125940,31.125381,31.126500,13.0
175898,175898,"POLYGON ((301806.277 8002917.445, 301795.79 80...",-18.052853,-18.053266,-18.052441,31.126970,31.126410,31.127530,2.0


The population of each 1km grid is distributed to underlying 100m cells proportionally based on building density. Each 100m grid is assigned a weight equal to its share of the total building count within the 1km grid.

In [18]:
# Adding population data at 1km grid to finer grid

data_path = Path(data_inputs)

# Load coarse population raster
pop_file = data_path / 'harare_zwe_f_15_49_2015_1km.tif'
pop_raster, transform, crs = read_tif(pop_file)

# Convert raster to vector population grid
pop_grid = raster2vector(pop_raster, transform, crs)
pop_grid = pop_grid.to_crs(epsg)
pop_grid['pop_grid_id'] = range(len(pop_grid))
pop_grid.to_csv(data_path / 'pop_grid_id.csv')

# Assign coarse population data to finer grid based on the centroid locations of the finer grid cells
grid['centroid'] = grid['geometry'].centroid
grid = gpd.sjoin(grid.set_geometry('centroid'), pop_grid, how='left', predicate='within')
print(grid.columns)
grid = grid[['grid_id', 'bcount', 'pop_grid_id', 'geometry', 'latitude', 'lat_min', 'lat_max',
       'longitude', 'lon_min', 'lon_max']]
grid.head()

Index(['grid_id', 'geometry', 'latitude', 'lat_min', 'lat_max', 'longitude',
       'lon_min', 'lon_max', 'bcount', 'centroid', 'index_right',
       'pop_grid_pop', 'pop_grid_id'],
      dtype='object')


,grid_id,bcount,pop_grid_id,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,16.0,41,"POLYGON ((302882.989 8053010.803, 302872.771 8...",-17.600408,-17.60082,-17.599996,31.141838,31.141280,31.142395
1,1,0.0,41,"POLYGON ((302992.081 8053011.873, 302981.862 8...",-17.600408,-17.60082,-17.599996,31.142865,31.142308,31.143423
2,2,0.0,41,"POLYGON ((303101.172 8053012.942, 303090.954 8...",-17.600408,-17.60082,-17.599996,31.143893,31.143335,31.144451
3,3,0.0,41,"POLYGON ((303210.263 8053014.01, 303200.046 80...",-17.600408,-17.60082,-17.599996,31.144921,31.144363,31.145479
4,4,1.0,42,"POLYGON ((303319.355 8053015.078, 303309.137 8...",-17.600408,-17.60082,-17.599996,31.145949,31.145391,31.146506


In [19]:
# Calculate population weight (fraction of total population count that should be assigned to cell based on its building count)
grid_grouped_pop = grid.groupby('pop_grid_id')
building_count_pop = grid_grouped_pop['bcount'].sum().rename('pop_grid_bcount')
grid = grid.merge(building_count_pop, on='pop_grid_id', how='left')
grid['pop_weight'] = grid['bcount'] / grid['pop_grid_bcount']

# Compute disaggregated population count based on weight and building count at coarser cell level
grid = grid.merge(pop_grid, on='pop_grid_id', how='left')
grid['pop'] = grid['pop_grid_pop'] * grid['pop_weight']
grid.head()

,grid_id,bcount,pop_grid_id,geometry_x,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,geometry_y,pop
0,0,16.0,41,"POLYGON ((302882.989 8053010.803, 302872.771 8...",-17.600408,-17.60082,-17.599996,31.141838,31.141280,31.142395,16.0,1.000000,NaN,"POLYGON ((302309.624 8053972.267, 303194.201 8...",NaN
1,1,0.0,41,"POLYGON ((302992.081 8053011.873, 302981.862 8...",-17.600408,-17.60082,-17.599996,31.142865,31.142308,31.143423,16.0,0.000000,NaN,"POLYGON ((302309.624 8053972.267, 303194.201 8...",NaN
2,2,0.0,41,"POLYGON ((303101.172 8053012.942, 303090.954 8...",-17.600408,-17.60082,-17.599996,31.143893,31.143335,31.144451,16.0,0.000000,NaN,"POLYGON ((302309.624 8053972.267, 303194.201 8...",NaN
3,3,0.0,41,"POLYGON ((303210.263 8053014.01, 303200.046 80...",-17.600408,-17.60082,-17.599996,31.144921,31.144363,31.145479,16.0,0.000000,NaN,"POLYGON ((302309.624 8053972.267, 303194.201 8...",NaN
4,4,1.0,42,"POLYGON ((303319.355 8053015.078, 303309.137 8...",-17.600408,-17.60082,-17.599996,31.145949,31.145391,31.146506,76.0,0.013158,NaN,"POLYGON ((303194.201 8053980.943, 304078.776 8...",NaN


In [20]:
# Saving to file
grid = grid.drop(columns=["geometry_y"])
# Keep all grid cells including those with zero population
grid["pop"] = (
    grid["pop"]
    .fillna(0)
)

In [21]:
grid = grid.set_geometry("geometry_x")
grid = grid.to_crs(4326)
grid.to_file(data_temp + 'pop-grid-harare.gpkg', driver='GPKG')

In [22]:
# Preparing gird centroids with population attribute for accessibility analysis
grid = gpd.read_file(data_temp + "pop-grid-harare.gpkg")
grid_ll = grid.to_crs(epsg=4326)

grid_ll["geometry"] = grid_ll.geometry.centroid

grid_ll["latitude"] = grid_ll["geometry"].y
grid_ll["longitude"] = grid_ll["geometry"].x

grid_centroids = grid_ll[["grid_id", "latitude", "longitude", "geometry", "pop"]].copy()
grid_centroids = grid_centroids.set_geometry("geometry")
grid_centroids.set_crs("EPSG:4326", inplace=True)

grid_centroids.reset_index(drop=True, inplace=True)

grid_centroids.to_file(data_temp + "grid_centroids.gpkg", driver="GPKG")

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_85583/4216881773.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  grid_ll["geometry"] = grid_ll.geometry.centroid


In [23]:
grid_centroids

,grid_id,latitude,longitude,geometry,pop
0,0,-17.600408,31.141838,POINT (31.14184 -17.60041),0.0
1,1,-17.600408,31.142865,POINT (31.14287 -17.60041),0.0
2,2,-17.600408,31.143893,POINT (31.14389 -17.60041),0.0
3,3,-17.600408,31.144921,POINT (31.14492 -17.60041),0.0
4,4,-17.600408,31.145949,POINT (31.14595 -17.60041),0.0
...,...,...,...,...,...
175895,175895,-18.052853,31.123882,POINT (31.12388 -18.05285),0.0
175896,175896,-18.052853,31.124911,POINT (31.12491 -18.05285),0.0
175897,175897,-18.052853,31.125940,POINT (31.12594 -18.05285),0.0
175898,175898,-18.052853,31.126970,POINT (31.12697 -18.05285),0.0


## 2. Spatial Analysis Pipeline

### Travel time and dista calculation using OpenRouteService (ORS)

Using OpenRouteService (ORS) Matrix API to calculate the travel time and distance from each population grid centroid to the healthcare facility. There are two options to process the time and distance calculations: Using the public ORS API or using a local instance of the ORS server.

note: this will generate a file 'OD_matrix_healthcare_pop_grid‘

In [3]:
origin_gdf = gpd.read_file(data_temp + "grid_centroids.gpkg")
origin_name_column = 'grid_id'
destination_gdf = gpd.read_file(data_inputs + 'healthcare_facilities_kisumu.geojson').dropna(subset=['geometry'])
destination_name_column = 'hcf_id'

In [4]:
# Extract coordinates
origins = list(zip(origin_gdf.geometry.x, origin_gdf.geometry.y))
destinations = list(zip(destination_gdf.geometry.x, destination_gdf.geometry.y))
locations = origins + destinations

In [5]:
# Indices
origins_index = list(range(0, len(origins)))
destinations_index = list(range(len(origins), len(locations)))

In [7]:
# Prepare API request
body = {
    'locations': locations,
    'destinations': destinations_index,
    'sources': origins_index,
    'metrics': ['distance', 'duration']
}

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': api_key,
    'Content-Type': 'application/json; charset=utf-8'
}

# Make request
response = requests.post(
    'https://api.openrouteservice.org/v2/matrix/driving-car',
    json=body,
    headers=headers
)

In [8]:
# Parse response
distances = response.json().get('distances', [])
durations = response.json().get('durations', [])

In [ ]:
distances_duration_matrix = []

# Iterate over each origin (grid)
for origin_index, origin in origin_gdf.iterrows():
    origin_name = origin[origin_name_column]
    origin_x = origin.geometry.x
    origin_y = origin.geometry.y
    origin_distances = distances[origin_index]
    origin_durations = durations[origin_index]

    # find the minimum duration and the index of the minimum duration
    min_duration = min(origin_durations)
    min_index = origin_durations.index(min_duration)
    destination_index = destinations_index[min_index]
    dest_x, dest_y = locations[destination_index]
    filtered = healthcare_facilities_validated[(destination_gdf.geometry.x == dest_x) & (destination_gdf.geometry.y == dest_y) ]
    destination_row = filtered.iloc[0]
    dest_name = destination_row[destination_name_column]

        # Append both the distance and duration for this origin-destination pair
    distances_duration_matrix.append([
            origin_name, origin_y, origin_x,
            dest_name, dest_y, dest_x,
            min_duration
        ])

In [ ]:
# Convert the results into a DataFrame
matrix_df = pd.DataFrame(distances_duration_matrix, columns=[
    'grid_code','origin_lat', 'origin_lon',
    'destination_name', 'dest_lat', 'dest_lon','min_duration'
])

In [ ]:
# Save to CSV
merged_df = pd.merge(matrix_df, grid_df[['grid_code', 'population']], on='grid_code', how='left')
merged_df.to_csv(data_temp + 'distance_duration_matrix_temp.csv', index=False)

In [ ]:
geometry = [Point(xy) for xy in zip(merged_df['dest_lon'], merged_df['dest_lat'])]
gdf = gpd.GeoDataFrame(merged_df, geometry=geometry, crs="EPSG:4326")

gpkg_path = data_temp + 'distance_duration_matrix_temp.gpkg'
gdf.to_file(gpkg_path, layer="duration_matrix", driver="GPKG")

### Option 2: Using a local ORS service
Make sure you have set a local service that runs the OSM-based ORS API. 
```r
# Insert R code from the local ORS service
```

### Procedure for Computing the OD Matrix Using a Local Docker Environment

This section outlines the steps required to compute the Origin-Destination (OD) matrix using a local Docker environment. 

1. **Set Up Docker Environment**:

2. **Prepare Input Data**:

3. **Run the OD Matrix Computation Script**:

4. **Monitor the Process**:

5. **Retrieve and Validate Output**:

### Diego please add description here

## Processing OD Matrix

Population data is the result of combining 1km grid data with 100m grid data. See [Section 2]() for more details.

In [30]:
# If not loaded yet, read from the temporary folder
centroids_df = gpd.read_file(data_temp +'pop-grid-harare.gpkg')
centroids_df

,grid_id,bcount,pop_grid_id,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,pop,geometry
0,0,16.0,41,-17.600408,-17.600820,-17.599996,31.141838,31.141280,31.142395,16.0,1.000000,NaN,0.0,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3..."
1,1,0.0,41,-17.600408,-17.600820,-17.599996,31.142865,31.142308,31.143423,16.0,0.000000,NaN,0.0,"POLYGON ((31.14342 -17.60082, 31.14334 -17.6, ..."
2,2,0.0,41,-17.600408,-17.600820,-17.599996,31.143893,31.143335,31.144451,16.0,0.000000,NaN,0.0,"POLYGON ((31.14445 -17.60082, 31.14436 -17.6, ..."
3,3,0.0,41,-17.600408,-17.600820,-17.599996,31.144921,31.144363,31.145479,16.0,0.000000,NaN,0.0,"POLYGON ((31.14548 -17.60082, 31.14539 -17.6, ..."
4,4,1.0,42,-17.600408,-17.600820,-17.599996,31.145949,31.145391,31.146506,76.0,0.013158,NaN,0.0,"POLYGON ((31.14651 -17.60082, 31.14642 -17.6, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175895,175895,0.0,3394,-18.052853,-18.053266,-18.052441,31.123882,31.123322,31.124441,86.0,0.000000,NaN,0.0,"POLYGON ((31.12444 -18.05327, 31.12435 -18.052..."
175896,175896,0.0,3394,-18.052853,-18.053266,-18.052441,31.124911,31.124351,31.125471,86.0,0.000000,NaN,0.0,"POLYGON ((31.12547 -18.05327, 31.12538 -18.052..."
175897,175897,13.0,3394,-18.052853,-18.053266,-18.052441,31.125940,31.125381,31.126500,86.0,0.151163,NaN,0.0,"POLYGON ((31.1265 -18.05327, 31.12641 -18.0524..."
175898,175898,2.0,3394,-18.052853,-18.053266,-18.052441,31.126970,31.126410,31.127530,86.0,0.023256,NaN,0.0,"POLYGON ((31.12753 -18.05327, 31.12744 -18.052..."


In [31]:
# If not loaded yet, read from the temporary folder
matrix_df = pd.read_csv(data_temp +'harare_access.csv')
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,0,0.0,2413.58,40.35
1,0,1.0,2418.62,40.36
2,0,2.0,2400.80,40.14
3,0,3.0,2400.13,40.17
4,0,4.0,2407.51,40.23
...,...,...,...,...
3693895,20,175895.0,2436.51,40.17
3693896,20,175896.0,2421.84,40.11
3693897,20,175897.0,2424.37,39.81
3693898,20,175898.0,2362.21,39.76


**GRID CELLS WITHOUT TRAVEL TIME ESTIMATE**

If a grid cell has a NULL value in the travel estimate, we will remove it from the analysis. This is because we cannot calculate the 2SFCA without a travel time estimate.

In [32]:
# Removing rows with NaN values in the 'duration_seconds' column
matrix_df = matrix_df.dropna(subset=['duration_seconds'])
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,0,0.0,2413.58,40.35
1,0,1.0,2418.62,40.36
2,0,2.0,2400.80,40.14
3,0,3.0,2400.13,40.17
4,0,4.0,2407.51,40.23
...,...,...,...,...
3693895,20,175895.0,2436.51,40.17
3693896,20,175896.0,2421.84,40.11
3693897,20,175897.0,2424.37,39.81
3693898,20,175898.0,2362.21,39.76


To process the OD Matrix we need merge it to create an integrated dataset that combines data from the healthcare facilities and population grid.For doing so, we will use the pandas library and join functions based on the id columns of all datasets.

In [33]:
pop_centroids_hcf = pd.merge(matrix_df, centroids_df[['grid_id', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max', 'lat_max','bcount','pop_grid_bcount', 'pop_grid_pop', 'pop', 'geometry']], 
                     left_on='destination_id', right_on='grid_id', how='left')
pop_centroids_hcf

,origin_id,destination_id,duration_seconds,distance_km,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,bcount,pop_grid_bcount,pop_grid_pop,pop,geometry
0,0,0.0,2413.58,40.35,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,16.0,16.0,NaN,0.0,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3..."
1,0,1.0,2418.62,40.36,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,0.0,16.0,NaN,0.0,"POLYGON ((31.14342 -17.60082, 31.14334 -17.6, ..."
2,0,2.0,2400.80,40.14,2,31.143893,-17.600408,31.143335,-17.600820,31.144451,-17.599996,0.0,16.0,NaN,0.0,"POLYGON ((31.14445 -17.60082, 31.14436 -17.6, ..."
3,0,3.0,2400.13,40.17,3,31.144921,-17.600408,31.144363,-17.600820,31.145479,-17.599996,0.0,16.0,NaN,0.0,"POLYGON ((31.14548 -17.60082, 31.14539 -17.6, ..."
4,0,4.0,2407.51,40.23,4,31.145949,-17.600408,31.145391,-17.600820,31.146506,-17.599996,1.0,76.0,NaN,0.0,"POLYGON ((31.14651 -17.60082, 31.14642 -17.6, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3693895,20,175895.0,2436.51,40.17,175895,31.123882,-18.052853,31.123322,-18.053266,31.124441,-18.052441,0.0,86.0,NaN,0.0,"POLYGON ((31.12444 -18.05327, 31.12435 -18.052..."
3693896,20,175896.0,2421.84,40.11,175896,31.124911,-18.052853,31.124351,-18.053266,31.125471,-18.052441,0.0,86.0,NaN,0.0,"POLYGON ((31.12547 -18.05327, 31.12538 -18.052..."
3693897,20,175897.0,2424.37,39.81,175897,31.125940,-18.052853,31.125381,-18.053266,31.126500,-18.052441,13.0,86.0,NaN,0.0,"POLYGON ((31.1265 -18.05327, 31.12641 -18.0524..."
3693898,20,175898.0,2362.21,39.76,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,2.0,86.0,NaN,0.0,"POLYGON ((31.12753 -18.05327, 31.12744 -18.052..."


In [34]:
pop_centroids_hcf = pop_centroids_hcf.rename(columns={
    "longitude": "origin_lon",
    "latitude": "origin_lat",
    "lon_min": "origin_lon_min",
    "lat_min": "origin_lat_min",
    "lon_max": "origin_lon_max",
    "lat_max": "origin_lat_max",
    "origin_id": "hcf_id",
    "pop": "population"
})
columns_to_keep = ["grid_id", "origin_lon", "origin_lat", "origin_lon_min","origin_lat_min","origin_lon_max","origin_lat_max","population", "bcount","pop_grid_bcount", "pop_grid_pop","geometry", "hcf_id", "duration_seconds", "distance_km"]
pop_centroids_hcf = pop_centroids_hcf[columns_to_keep]

In [35]:
pop_centroids_hcf

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,hcf_id,duration_seconds,distance_km
0,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,NaN,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3...",0,2413.58,40.35
1,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14342 -17.60082, 31.14334 -17.6, ...",0,2418.62,40.36
2,2,31.143893,-17.600408,31.143335,-17.600820,31.144451,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14445 -17.60082, 31.14436 -17.6, ...",0,2400.80,40.14
3,3,31.144921,-17.600408,31.144363,-17.600820,31.145479,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14548 -17.60082, 31.14539 -17.6, ...",0,2400.13,40.17
4,4,31.145949,-17.600408,31.145391,-17.600820,31.146506,-17.599996,0.0,1.0,76.0,NaN,"POLYGON ((31.14651 -17.60082, 31.14642 -17.6, ...",0,2407.51,40.23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3693895,175895,31.123882,-18.052853,31.123322,-18.053266,31.124441,-18.052441,0.0,0.0,86.0,NaN,"POLYGON ((31.12444 -18.05327, 31.12435 -18.052...",20,2436.51,40.17
3693896,175896,31.124911,-18.052853,31.124351,-18.053266,31.125471,-18.052441,0.0,0.0,86.0,NaN,"POLYGON ((31.12547 -18.05327, 31.12538 -18.052...",20,2421.84,40.11
3693897,175897,31.125940,-18.052853,31.125381,-18.053266,31.126500,-18.052441,0.0,13.0,86.0,NaN,"POLYGON ((31.1265 -18.05327, 31.12641 -18.0524...",20,2424.37,39.81
3693898,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,0.0,2.0,86.0,NaN,"POLYGON ((31.12753 -18.05327, 31.12744 -18.052...",20,2362.21,39.76


Merging the dataframe than contains the od matrix (with the healthcare facility class) and the population data with the full information about health care facilities.

In [36]:
healthcare_facilities_validated

,Longitude,Latitude,Facility_name,Province,District,Ward,Ownership_type,Ownership,EmOC_status,Services_offered,Facility_level,Facility_type,geometry,local_validation,hcf_id
13,30.994402,-17.884174,Highfields Polyclinic,Harare,Harare,25,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (30.9944 -17.88417),Public Basic EmOC,0
16,31.035328,-17.859606,Mbare Polyclinic,Harare,Harare,11,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (31.03533 -17.85961),Public Basic EmOC,1
22,30.950854,-17.907724,Glenview Polyclinic,Harare,Harare,31,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (30.95085 -17.90772),Public Basic EmOC,2
29,31.062272,-18.017600,Chitungwiza General Hospital,Harare,Chitungwiza,13,Public,Government / Ministry of Health and Child Care...,Comprehensive,Full obstetric & surgical services,Tertiary,General hospital,POINT (31.06227 -18.0176),Public Comprehensive EmOC,3
38,30.970587,-17.854463,Kambuzuma Polyclinic,Harare,Harare,14,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (30.97059 -17.85446),Public Basic EmOC,4
42,31.196645,-17.835547,Mabvuku Polyclinic,Harare,Harare,20,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (31.19664 -17.83555),Public Basic EmOC,5
45,31.086431,-17.878907,Hatfield Polyclinic,Harare,Harare,22,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (31.08643 -17.87891),Public Basic EmOC,6
50,31.107381,-17.697793,Hatcliffe Polyclinic,Harare,Harare,Peri-urban/Informal,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (31.10738 -17.69779),Public Basic EmOC,7
52,31.012382,-17.860447,Harare Central Hospital,Harare,Harare,13,Public,Government / Ministry of Health and Child Care...,Comprehensive,Full obstetric & surgical services,Quaternary,Central hospital,POINT (31.01238 -17.86045),Public Comprehensive EmOC,8
55,30.935376,-17.891499,Budiriro Polyclinic,Harare,Harare,33,Public,Local council,Basic,"Outpatient, maternity, and minor surgical serv...",Secondary,Polyclinic,POINT (30.93538 -17.8915),Public Basic EmOC,9


In [37]:
distances_duration_matrix = pd.merge(pop_centroids_hcf, healthcare_facilities_validated[['hcf_id','Facility_name', 'local_validation']], 
                     left_on='hcf_id', right_on='hcf_id', how='left') # left on is from od matrix, right on is from healthcare facilities

In [38]:
distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,hcf_id,duration_seconds,distance_km,Facility_name,local_validation
0,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,NaN,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3...",0,2413.58,40.35,Highfields Polyclinic,Public Basic EmOC
1,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14342 -17.60082, 31.14334 -17.6, ...",0,2418.62,40.36,Highfields Polyclinic,Public Basic EmOC
2,2,31.143893,-17.600408,31.143335,-17.600820,31.144451,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14445 -17.60082, 31.14436 -17.6, ...",0,2400.80,40.14,Highfields Polyclinic,Public Basic EmOC
3,3,31.144921,-17.600408,31.144363,-17.600820,31.145479,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14548 -17.60082, 31.14539 -17.6, ...",0,2400.13,40.17,Highfields Polyclinic,Public Basic EmOC
4,4,31.145949,-17.600408,31.145391,-17.600820,31.146506,-17.599996,0.0,1.0,76.0,NaN,"POLYGON ((31.14651 -17.60082, 31.14642 -17.6, ...",0,2407.51,40.23,Highfields Polyclinic,Public Basic EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3693895,175895,31.123882,-18.052853,31.123322,-18.053266,31.124441,-18.052441,0.0,0.0,86.0,NaN,"POLYGON ((31.12444 -18.05327, 31.12435 -18.052...",20,2436.51,40.17,Dr. D Djordjevic,Private Comprehensive EmOC
3693896,175896,31.124911,-18.052853,31.124351,-18.053266,31.125471,-18.052441,0.0,0.0,86.0,NaN,"POLYGON ((31.12547 -18.05327, 31.12538 -18.052...",20,2421.84,40.11,Dr. D Djordjevic,Private Comprehensive EmOC
3693897,175897,31.125940,-18.052853,31.125381,-18.053266,31.126500,-18.052441,0.0,13.0,86.0,NaN,"POLYGON ((31.1265 -18.05327, 31.12641 -18.0524...",20,2424.37,39.81,Dr. D Djordjevic,Private Comprehensive EmOC
3693898,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,0.0,2.0,86.0,NaN,"POLYGON ((31.12753 -18.05327, 31.12744 -18.052...",20,2362.21,39.76,Dr. D Djordjevic,Private Comprehensive EmOC


In [39]:
category_counts = healthcare_facilities_validated['local_validation'].value_counts()
print(category_counts)

local_validation
Public Basic EmOC             17
Public Comprehensive EmOC      3
Private Comprehensive EmOC     1
Name: count, dtype: int64


In [40]:
selected_categories = ['Public Comprehensive EmOC', 'Private Comprehensive EmOC', 'Public Basic EmOC']

In [41]:
distances_duration_matrix = distances_duration_matrix[
    distances_duration_matrix['local_validation'].isin(selected_categories)]

distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,hcf_id,duration_seconds,distance_km,Facility_name,local_validation
0,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,NaN,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3...",0,2413.58,40.35,Highfields Polyclinic,Public Basic EmOC
1,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14342 -17.60082, 31.14334 -17.6, ...",0,2418.62,40.36,Highfields Polyclinic,Public Basic EmOC
2,2,31.143893,-17.600408,31.143335,-17.600820,31.144451,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14445 -17.60082, 31.14436 -17.6, ...",0,2400.80,40.14,Highfields Polyclinic,Public Basic EmOC
3,3,31.144921,-17.600408,31.144363,-17.600820,31.145479,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14548 -17.60082, 31.14539 -17.6, ...",0,2400.13,40.17,Highfields Polyclinic,Public Basic EmOC
4,4,31.145949,-17.600408,31.145391,-17.600820,31.146506,-17.599996,0.0,1.0,76.0,NaN,"POLYGON ((31.14651 -17.60082, 31.14642 -17.6, ...",0,2407.51,40.23,Highfields Polyclinic,Public Basic EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3693895,175895,31.123882,-18.052853,31.123322,-18.053266,31.124441,-18.052441,0.0,0.0,86.0,NaN,"POLYGON ((31.12444 -18.05327, 31.12435 -18.052...",20,2436.51,40.17,Dr. D Djordjevic,Private Comprehensive EmOC
3693896,175896,31.124911,-18.052853,31.124351,-18.053266,31.125471,-18.052441,0.0,0.0,86.0,NaN,"POLYGON ((31.12547 -18.05327, 31.12538 -18.052...",20,2421.84,40.11,Dr. D Djordjevic,Private Comprehensive EmOC
3693897,175897,31.125940,-18.052853,31.125381,-18.053266,31.126500,-18.052441,0.0,13.0,86.0,NaN,"POLYGON ((31.1265 -18.05327, 31.12641 -18.0524...",20,2424.37,39.81,Dr. D Djordjevic,Private Comprehensive EmOC
3693898,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,0.0,2.0,86.0,NaN,"POLYGON ((31.12753 -18.05327, 31.12744 -18.052...",20,2362.21,39.76,Dr. D Djordjevic,Private Comprehensive EmOC


In [43]:
# creat subsets based on categories of 'Validation of HCFs Categorization'
categories = {
    "public_comprehensive_EmOC": ["Public Comprehensive EmOC"],
    "private_comprehensive_EmOC": ["Private Comprehensive EmOC"],
    "public_basic_EmOC": ["Public Basic EmOC"]
}

subsets = {
    key: distances_duration_matrix[
        distances_duration_matrix['local_validation'].str.contains('|'.join(values), na=False)
    ]
    for key, values in categories.items()
}

public_CEmOC = subsets["public_comprehensive_EmOC"]
private_CEmOC = subsets["private_comprehensive_EmOC"]
public_BEmOC = subsets["public_basic_EmOC"]

In [44]:
# Step 2: Define a function to get 3 smallest duration_seconds per grid_id for each category
def get_closest_3(df, n=3):
    return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_seconds')).reset_index(drop=True)

In [45]:
# Step 3: If the subsets are already created for each category, we apply the function to each subset:
public_CEmOC_closest_3 = get_closest_3(public_CEmOC)
private_CEmOC_closest_3 = get_closest_3(private_CEmOC)
public_BEmOC_closest_3 = get_closest_3(public_BEmOC)

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_85583/1217296076.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_seconds')).reset_index(drop=True)
/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_85583/1217296076.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsm

In [46]:
# Step 4: Concatenate the filtered results into a single DataFrame
distances_duration_matrix = pd.concat([
    public_CEmOC_closest_3, private_CEmOC_closest_3,
    public_BEmOC_closest_3
])
distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,hcf_id,duration_seconds,distance_km,Facility_name,local_validation
0,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,NaN,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3...",13,1854.61,29.67,Parirenyatwa Hospital,Public Comprehensive EmOC
1,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,NaN,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3...",8,2277.27,37.19,Harare Central Hospital,Public Comprehensive EmOC
2,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,NaN,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3...",3,3300.24,54.32,Chitungwiza General Hospital,Public Comprehensive EmOC
3,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14342 -17.60082, 31.14334 -17.6, ...",13,1859.66,29.68,Parirenyatwa Hospital,Public Comprehensive EmOC
4,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14342 -17.60082, 31.14334 -17.6, ...",8,2282.31,37.20,Harare Central Hospital,Public Comprehensive EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527695,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,0.0,2.0,86.0,NaN,"POLYGON ((31.12753 -18.05327, 31.12744 -18.052...",19,1905.93,30.96,Glennorah Polyclinic,Public Basic EmOC
527696,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,0.0,2.0,86.0,NaN,"POLYGON ((31.12753 -18.05327, 31.12744 -18.052...",11,1923.54,30.95,Rutsanana Maternity Clinic,Public Basic EmOC
527697,175899,31.127999,-18.052853,31.127439,-18.053266,31.128559,-18.052441,0.0,3.0,86.0,NaN,"POLYGON ((31.12856 -18.05327, 31.12847 -18.052...",6,1623.77,27.93,Hatfield Polyclinic,Public Basic EmOC
527698,175899,31.127999,-18.052853,31.127439,-18.053266,31.128559,-18.052441,0.0,3.0,86.0,NaN,"POLYGON ((31.12856 -18.05327, 31.12847 -18.052...",19,1895.67,30.88,Glennorah Polyclinic,Public Basic EmOC


In [47]:
geometry = [Point(xy) for xy in zip(distances_duration_matrix['origin_lon'], distances_duration_matrix['origin_lat'])]
gdf = gpd.GeoDataFrame(distances_duration_matrix, geometry=geometry, crs="EPSG:4326")

In [48]:
gpkg_path = data_temp + "distances_duration_3_closet_Emoc.gpkg"
gdf.to_file(gpkg_path, layer="distances_duration_3_closet_Emoc", driver="GPKG", mode="w")

In [49]:
# Review and remove
origin_dest = distances_duration_matrix

## Enhanced Two-Step Floating Catchment Area (E2SFCA) method

In [50]:
# Function
from math import *
d = 10 * 60 # try max duration 5/10mins/15mins/20 car, under estimation of travel time and traffic condition realted to the selected data sourse 
W = 0.01
beta = - d ** 2 / log(W)
print(beta)

78173.00674258533


In [51]:
print(origin_dest.head())

   grid_id  origin_lon  origin_lat  origin_lon_min  origin_lat_min  \
0        0   31.141838  -17.600408       31.141280       -17.60082   
1        0   31.141838  -17.600408       31.141280       -17.60082   
2        0   31.141838  -17.600408       31.141280       -17.60082   
3        1   31.142865  -17.600408       31.142308       -17.60082   
4        1   31.142865  -17.600408       31.142308       -17.60082   

   origin_lon_max  origin_lat_max  population  bcount  pop_grid_bcount  \
0       31.142395      -17.599996         0.0    16.0             16.0   
1       31.142395      -17.599996         0.0    16.0             16.0   
2       31.142395      -17.599996         0.0    16.0             16.0   
3       31.143423      -17.599996         0.0     0.0             16.0   
4       31.143423      -17.599996         0.0     0.0             16.0   

   pop_grid_pop                                           geometry  hcf_id  \
0           NaN  POLYGON ((31.1424 -17.60082, 31.14231 -

In [52]:
# Convert 'duration' to numeric, coercing errors to NaN
origin_dest = origin_dest.copy()
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')

In [53]:
# Drop rows with NaN values in 'duration' column
origin_dest = origin_dest.dropna(subset=['duration_seconds'])
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')
origin_dest_acc = origin_dest  # Backup

In [54]:
# Apply Gaussian decay function to calculate the weight of each grid to healthcare 
# facilities based on the travel duration. d is the travel time and beta is the decay 
# parameter previously calculated.
# The weight decreases as the duration increases, meaning facilities that are further away have less impact.
origin_dest_acc['Weight'] = origin_dest_acc['duration_seconds'].apply(lambda d: round(math.exp(-d**2/beta), 8))

In [55]:
# Compute the Weighted Population (Pop_W), the population of each grid cell is multiplied 
# by the corresponding weight to calculate the weighted population.
origin_dest_acc['Pop_W'] = origin_dest_acc['population'] * origin_dest_acc['Weight']

In [56]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,hcf_id,duration_seconds,distance_km,Facility_name,local_validation,Weight,Pop_W
0,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,NaN,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3...",13,1854.61,29.67,Parirenyatwa Hospital,Public Comprehensive EmOC,0.0,0.0
1,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,NaN,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3...",8,2277.27,37.19,Harare Central Hospital,Public Comprehensive EmOC,0.0,0.0
2,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,NaN,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3...",3,3300.24,54.32,Chitungwiza General Hospital,Public Comprehensive EmOC,0.0,0.0
3,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14342 -17.60082, 31.14334 -17.6, ...",13,1859.66,29.68,Parirenyatwa Hospital,Public Comprehensive EmOC,0.0,0.0
4,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14342 -17.60082, 31.14334 -17.6, ...",8,2282.31,37.20,Harare Central Hospital,Public Comprehensive EmOC,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527695,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,0.0,2.0,86.0,NaN,"POLYGON ((31.12753 -18.05327, 31.12744 -18.052...",19,1905.93,30.96,Glennorah Polyclinic,Public Basic EmOC,0.0,0.0
527696,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,0.0,2.0,86.0,NaN,"POLYGON ((31.12753 -18.05327, 31.12744 -18.052...",11,1923.54,30.95,Rutsanana Maternity Clinic,Public Basic EmOC,0.0,0.0
527697,175899,31.127999,-18.052853,31.127439,-18.053266,31.128559,-18.052441,0.0,3.0,86.0,NaN,"POLYGON ((31.12856 -18.05327, 31.12847 -18.052...",6,1623.77,27.93,Hatfield Polyclinic,Public Basic EmOC,0.0,0.0
527698,175899,31.127999,-18.052853,31.127439,-18.053266,31.128559,-18.052441,0.0,3.0,86.0,NaN,"POLYGON ((31.12856 -18.05327, 31.12847 -18.052...",19,1895.67,30.88,Glennorah Polyclinic,Public Basic EmOC,0.0,0.0


In [57]:
# Sum the Weighted Population
origin_dest_sum = origin_dest_acc.groupby(by='hcf_id')['Pop_W'].sum().reset_index()

In [58]:
origin_dest_sum

,hcf_id,Pop_W
0,0,6592.589022
1,1,10594.972205
2,2,9223.105133
3,3,24628.292227
4,4,8218.617333
5,5,5971.117643
6,6,15588.025668
7,7,6809.456859
8,8,5584.175417
9,9,6513.663089


In [59]:
# Merge the Sum of Weighted Population Back into the Original Data
origin_dest_acc = origin_dest_acc.merge(origin_dest_sum, on='hcf_id')

In [60]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,hcf_id,duration_seconds,distance_km,Facility_name,local_validation,Weight,Pop_W_x,Pop_W_y
0,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,NaN,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3...",13,1854.61,29.67,Parirenyatwa Hospital,Public Comprehensive EmOC,0.0,0.0,10120.879363
1,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,NaN,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3...",8,2277.27,37.19,Harare Central Hospital,Public Comprehensive EmOC,0.0,0.0,5584.175417
2,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,NaN,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3...",3,3300.24,54.32,Chitungwiza General Hospital,Public Comprehensive EmOC,0.0,0.0,24628.292227
3,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14342 -17.60082, 31.14334 -17.6, ...",13,1859.66,29.68,Parirenyatwa Hospital,Public Comprehensive EmOC,0.0,0.0,10120.879363
4,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,0.0,0.0,16.0,NaN,"POLYGON ((31.14342 -17.60082, 31.14334 -17.6, ...",8,2282.31,37.20,Harare Central Hospital,Public Comprehensive EmOC,0.0,0.0,5584.175417
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1231295,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,0.0,2.0,86.0,NaN,"POLYGON ((31.12753 -18.05327, 31.12744 -18.052...",19,1905.93,30.96,Glennorah Polyclinic,Public Basic EmOC,0.0,0.0,5249.119051
1231296,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,0.0,2.0,86.0,NaN,"POLYGON ((31.12753 -18.05327, 31.12744 -18.052...",11,1923.54,30.95,Rutsanana Maternity Clinic,Public Basic EmOC,0.0,0.0,4218.455130
1231297,175899,31.127999,-18.052853,31.127439,-18.053266,31.128559,-18.052441,0.0,3.0,86.0,NaN,"POLYGON ((31.12856 -18.05327, 31.12847 -18.052...",6,1623.77,27.93,Hatfield Polyclinic,Public Basic EmOC,0.0,0.0,15588.025668
1231298,175899,31.127999,-18.052853,31.127439,-18.053266,31.128559,-18.052441,0.0,3.0,86.0,NaN,"POLYGON ((31.12856 -18.05327, 31.12847 -18.052...",19,1895.67,30.88,Glennorah Polyclinic,Public Basic EmOC,0.0,0.0,5249.119051


In [61]:
# supply value is set to 1 for simplicity (capacity of HCF)
# supply = 1
# in the future, we will link supply with ownership and EmOC service level
origin_dest_acc = origin_dest_acc.rename(columns={'Pop_W_y': 'Pop_W_S'})  # Pop_W_S: Population Weight Sum

In [62]:
supply_map = {
    'Public Comprehensive EmOC': 1,
    'Private Comprehensive EmOC': 0.7,
    'Public Basic EmOC': 0.5
}

In [64]:
origin_dest_acc['supply'] = origin_dest_acc['local_validation'].map(supply_map)
origin_dest_acc['supply_demand_ratio'] = origin_dest_acc['supply'] / origin_dest_acc['Pop_W_S']

origin_dest_acc['supply_demand_ratio'] = (
    origin_dest_acc['supply_demand_ratio']
        .replace([np.inf, -np.inf, np.nan], 0)
)


In [65]:
# Calculate Rj * Weight for Each Grid Cell
origin_dest_acc['supply_W'] = origin_dest_acc['supply_demand_ratio'] * origin_dest_acc.Weight

In [66]:
# Compute Accessibility Index (Ai) for Each Grid Cell
origin_dest_acc['Accessibility'] = origin_dest_acc.groupby('grid_id')['supply_W'].transform('sum')

In [67]:
# Normalize
scaler = MinMaxScaler()
origin_dest_acc['Accessibility_standard'] = scaler.fit_transform(origin_dest_acc[['Accessibility']])

In [68]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,Facility_name,local_validation,Weight,Pop_W_x,Pop_W_S,supply,supply_demand_ratio,supply_W,Accessibility,Accessibility_standard
0,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,...,Parirenyatwa Hospital,Public Comprehensive EmOC,0.0,0.0,10120.879363,1.0,0.000099,0.0,1.076224e-08,3.120677e-05
1,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,...,Harare Central Hospital,Public Comprehensive EmOC,0.0,0.0,5584.175417,1.0,0.000179,0.0,1.076224e-08,3.120677e-05
2,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,0.0,16.0,16.0,...,Chitungwiza General Hospital,Public Comprehensive EmOC,0.0,0.0,24628.292227,1.0,0.000041,0.0,1.076224e-08,3.120677e-05
3,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,0.0,0.0,16.0,...,Parirenyatwa Hospital,Public Comprehensive EmOC,0.0,0.0,10120.879363,1.0,0.000099,0.0,9.665969e-09,2.802797e-05
4,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,0.0,0.0,16.0,...,Harare Central Hospital,Public Comprehensive EmOC,0.0,0.0,5584.175417,1.0,0.000179,0.0,9.665969e-09,2.802797e-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1231295,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,0.0,2.0,86.0,...,Glennorah Polyclinic,Public Basic EmOC,0.0,0.0,5249.119051,0.5,0.000095,0.0,2.192600e-11,6.357782e-08
1231296,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,0.0,2.0,86.0,...,Rutsanana Maternity Clinic,Public Basic EmOC,0.0,0.0,4218.455130,0.5,0.000119,0.0,2.192600e-11,6.357782e-08
1231297,175899,31.127999,-18.052853,31.127439,-18.053266,31.128559,-18.052441,0.0,3.0,86.0,...,Hatfield Polyclinic,Public Basic EmOC,0.0,0.0,15588.025668,0.5,0.000032,0.0,2.923467e-11,8.477043e-08
1231298,175899,31.127999,-18.052853,31.127439,-18.053266,31.128559,-18.052441,0.0,3.0,86.0,...,Glennorah Polyclinic,Public Basic EmOC,0.0,0.0,5249.119051,0.5,0.000095,0.0,2.923467e-11,8.477043e-08


In [69]:
max(origin_dest_acc.Accessibility_standard)

1.0

In [70]:
gdf = gpd.GeoDataFrame(origin_dest_acc, geometry='geometry', crs="EPSG:4326")
gpkg_path = data_temp + 'acc_score_3closest.gpkg'
gdf.to_file(gpkg_path, layer="acc_score_3closest", driver="GPKG")

# 4. Grouping by grid ID to prepare the final output file
There is a need to update this part of the code

In [71]:
# Read the GeoPackage file (if starting from this section)
results_grid = gpd.read_file(data_temp + 'acc_score_3closest.gpkg')

In [72]:
results_grid = results_grid[['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry']]

In [73]:
# Group by multiple columns and calculate the mean for numeric columns
# results_grid = results_grid.groupby(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard']).count().reset_index()
results_grid = results_grid.drop_duplicates(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry'])
type(results_grid)

geopandas.geodataframe.GeoDataFrame

In [74]:
# save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access.gpkg'
results_grid.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access', driver='GPKG')

In [75]:
results_grid

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,Accessibility_standard,geometry
0,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,3.120677e-05,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3..."
3,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,2.802797e-05,"POLYGON ((31.14342 -17.60082, 31.14334 -17.6, ..."
6,2,31.143893,-17.600408,31.143335,-17.600820,31.144451,-17.599996,4.086028e-05,"POLYGON ((31.14445 -17.60082, 31.14436 -17.6, ..."
9,3,31.144921,-17.600408,31.144363,-17.600820,31.145479,-17.599996,4.143728e-05,"POLYGON ((31.14548 -17.60082, 31.14539 -17.6, ..."
12,4,31.145949,-17.600408,31.145391,-17.600820,31.146506,-17.599996,3.548634e-05,"POLYGON ((31.14651 -17.60082, 31.14642 -17.6, ..."
...,...,...,...,...,...,...,...,...,...
527685,175895,31.123882,-18.052853,31.123322,-18.053266,31.124441,-18.052441,6.946466e-08,"POLYGON ((31.12444 -18.05327, 31.12435 -18.052..."
527688,175896,31.124911,-18.052853,31.124351,-18.053266,31.125471,-18.052441,4.709469e-08,"POLYGON ((31.12547 -18.05327, 31.12538 -18.052..."
527691,175897,31.125940,-18.052853,31.125381,-18.053266,31.126500,-18.052441,1.883787e-08,"POLYGON ((31.1265 -18.05327, 31.12641 -18.0524..."
527694,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,6.357782e-08,"POLYGON ((31.12753 -18.05327, 31.12744 -18.052..."


### Setting values for Low medium and High categories

We started by defining equal value division, and modified the thesholds to a value that is more legible and easier to interpret. Every model should have their own thresholds based on the data distribution of the three categories. 

Note: For Kano, we excluded grid cells with index values below 0.000001 that indicated very low population and a small number of buildings.  

In [76]:
results_grid['result'] = -1
results_grid.loc[results_grid['Accessibility_standard'] >= 0, 'result'] = 2
results_grid.loc[results_grid['Accessibility_standard'] > 0.0000026, 'result'] = 1
results_grid.loc[results_grid['Accessibility_standard'] > 0.0053046, 'result'] = 0

In [77]:
category_counts = results_grid['result'].value_counts()
print(category_counts)

result
1    58683
0    58633
2    58584
Name: count, dtype: int64


### Setting values for focus areas

We defined the focus areas based on values for the different thresholds. We aim at participants helping us to confirm the selection of the city-specific thresholds.

In [78]:
results_grid['focused'] = 0
# Focus areas between the Low category and the excluded cells due to low population or no buildings
results_grid.loc[(results_grid['Accessibility_standard'] > 0.0000001) & (results_grid['Accessibility_standard'] < 0.0000002), 'focused'] = 1
# Focus areas between the Medium and High categories
results_grid.loc[(results_grid['Accessibility_standard'] > 0.000001) & (results_grid['Accessibility_standard'] < 0.000005), 'focused'] = 1
# Focus areas between the Low and Medium categories
results_grid.loc[(results_grid['Accessibility_standard'] > 0.003) & (results_grid['Accessibility_standard'] < 0.007), 'focused'] = 1

In [79]:
results_grid = results_grid.loc[results_grid['result'] != -1]

In [80]:
results_grid = results_grid.rename(columns={
    'origin_lon': 'longitude',
    'origin_lat': 'latitude',
    'origin_lon_min': 'lon_min',
    'origin_lat_min': 'lat_min',
    'origin_lon_max': 'lon_max',
    'origin_lat_max': 'lat_max'
})

In [81]:
results_grid

,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,Accessibility_standard,geometry,result,focused
0,0,31.141838,-17.600408,31.141280,-17.600820,31.142395,-17.599996,3.120677e-05,"POLYGON ((31.1424 -17.60082, 31.14231 -17.6, 3...",1,0
3,1,31.142865,-17.600408,31.142308,-17.600820,31.143423,-17.599996,2.802797e-05,"POLYGON ((31.14342 -17.60082, 31.14334 -17.6, ...",1,0
6,2,31.143893,-17.600408,31.143335,-17.600820,31.144451,-17.599996,4.086028e-05,"POLYGON ((31.14445 -17.60082, 31.14436 -17.6, ...",1,0
9,3,31.144921,-17.600408,31.144363,-17.600820,31.145479,-17.599996,4.143728e-05,"POLYGON ((31.14548 -17.60082, 31.14539 -17.6, ...",1,0
12,4,31.145949,-17.600408,31.145391,-17.600820,31.146506,-17.599996,3.548634e-05,"POLYGON ((31.14651 -17.60082, 31.14642 -17.6, ...",1,0
...,...,...,...,...,...,...,...,...,...,...,...
527685,175895,31.123882,-18.052853,31.123322,-18.053266,31.124441,-18.052441,6.946466e-08,"POLYGON ((31.12444 -18.05327, 31.12435 -18.052...",2,0
527688,175896,31.124911,-18.052853,31.124351,-18.053266,31.125471,-18.052441,4.709469e-08,"POLYGON ((31.12547 -18.05327, 31.12538 -18.052...",2,0
527691,175897,31.125940,-18.052853,31.125381,-18.053266,31.126500,-18.052441,1.883787e-08,"POLYGON ((31.1265 -18.05327, 31.12641 -18.0524...",2,0
527694,175898,31.126970,-18.052853,31.126410,-18.053266,31.127530,-18.052441,6.357782e-08,"POLYGON ((31.12753 -18.05327, 31.12744 -18.052...",2,0


In [82]:
# Save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access-class.gpkg'
results_grid.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access-class', driver='GPKG')

In [84]:
# Save the results to a CSV file in the format required by the IDEAMAPS data ecosystem
results_table = results_grid.drop(columns=['Accessibility_standard', 'grid_id', 'geometry'])
results_table.to_csv(model_outputs + 'model-output.csv', index=False)